In [ ]:
# =========================
# open_nifti_folder_napari.py
# =========================

from pathlib import Path
import nibabel as nib
import numpy as np
import napari


# ------------------------------------------------------------
# EDIT THIS PATH ONLY
# ------------------------------------------------------------
NIFTI_FOLDER = Path(r'/Users/m298134/Desktop/MRI/Random Images/Anomalyv1.0/Final Cropped Brains/Brain Only 2 Brain Trial')

RECURSIVE = True       # True = search subfolders too
OPEN_IN_3D = True      # True = napari starts in 3D mode
NORMALIZE_IMAGES = True


def normalize_for_display(data, p_low=1.0, p_high=99.8):
    data = np.asarray(data, dtype=np.float32)
    finite = np.isfinite(data)

    if not finite.any():
        return np.zeros_like(data, dtype=np.float32)

    vals = data[finite]
    nonzero = vals[np.abs(vals) > 1e-8]
    if nonzero.size > 100:
        vals = nonzero

    lo = float(np.percentile(vals, p_low))
    hi = float(np.percentile(vals, p_high))

    if hi <= lo:
        return np.zeros_like(data, dtype=np.float32)

    out = (data - lo) / (hi - lo)
    out = np.clip(out, 0, 1)
    out[~finite] = 0
    return out.astype(np.float32)


def load_nifti_as_zyx(path):
    img = nib.load(str(path))
    data_xyz = img.get_fdata(dtype=np.float32)

    if data_xyz.ndim == 3:
        data_zyx = np.transpose(data_xyz, (2, 1, 0))
    elif data_xyz.ndim == 4:
        data_zyx = np.transpose(data_xyz, (3, 2, 1, 0))
    else:
        raise ValueError(f"Unsupported NIfTI dimensionality {data_xyz.ndim}: {path}")

    return data_zyx.astype(np.float32)


def looks_like_label(path, data):
    name = path.name.lower()

    label_keywords = [
        "mask",
        "label",
        "labels",
        "seg",
        "segmentation",
        "annotation",
        "roi",
        "binary",
        "candidate",
        "anomaly_mask",
        "detected",
    ]

    if any(k in name for k in label_keywords):
        return True

    finite = data[np.isfinite(data)]
    if finite.size == 0:
        return False

    unique_vals = np.unique(finite)
    if unique_vals.size <= 30 and np.allclose(unique_vals, np.round(unique_vals)):
        return True

    return False


def find_niftis(folder, recursive=True):
    if not folder.exists():
        raise FileNotFoundError(f"Folder does not exist:\n{folder}")

    if recursive:
        files = sorted(folder.rglob("*"))
    else:
        files = sorted(folder.glob("*"))

    niftis = [
        f for f in files
        if f.is_file() and f.name.lower().endswith((".nii", ".nii.gz"))
    ]

    return niftis


nifti_paths = find_niftis(NIFTI_FOLDER, recursive=RECURSIVE)

if len(nifti_paths) == 0:
    raise FileNotFoundError(f"No .nii or .nii.gz files found in:\n{NIFTI_FOLDER}")

print(f"Found {len(nifti_paths)} NIfTI files:")
for p in nifti_paths:
    print(" ", p)

viewer = napari.Viewer(ndisplay=3 if OPEN_IN_3D else 2)
viewer.title = f"NIfTI folder viewer - {NIFTI_FOLDER.name}"

for path in nifti_paths:
    data = load_nifti_as_zyx(path)

    if looks_like_label(path, data):
        viewer.add_labels(
            data.astype(np.uint32),
            name=path.name,
            visible=True,
        )
    else:
        display_data = normalize_for_display(data) if NORMALIZE_IMAGES else data
        viewer.add_image(
            display_data,
            name=path.name,
            opacity=0.75,
            blending="translucent",
            contrast_limits=(0, 1) if NORMALIZE_IMAGES else None,
            visible=True,
        )

napari.run()

Found 6 NIfTI files:
  /Users/m298134/Desktop/MRI/Random Images/Anomalyv1.0/Final Cropped Brains/Brain Only 2 Brain Trial/4_scan_2_FINAL_EDITED_pasteback_brain_only_full_head.nii
  /Users/m298134/Desktop/MRI/Random Images/Anomalyv1.0/Final Cropped Brains/Brain Only 2 Brain Trial/4_scan_2_FINAL_EDITED_pasteback_mask_full_head.nii
  /Users/m298134/Desktop/MRI/Random Images/Anomalyv1.0/Final Cropped Brains/Brain Only 2 Brain Trial/6_scan_2_FINAL_EDITED_pasteback_brain_only_full_head.nii
  /Users/m298134/Desktop/MRI/Random Images/Anomalyv1.0/Final Cropped Brains/Brain Only 2 Brain Trial/6_scan_2_FINAL_EDITED_pasteback_mask_full_head.nii
  /Users/m298134/Desktop/MRI/Random Images/Anomalyv1.0/Final Cropped Brains/Brain Only 2 Brain Trial/GL261ENLN_981_scan_3_cropped_brain_only.nii.gz
  /Users/m298134/Desktop/MRI/Random Images/Anomalyv1.0/Final Cropped Brains/Brain Only 2 Brain Trial/GL261ENLN_981_scan_3_cropped_mask.nii.gz


: 